# Publish the fine-tuned LFM2 Extract: the int8 CPU export

Starts from the merged model already on the Hub (no training), builds the int8 CPU export that is still missing from the ONNX repo, checks it, and uploads it.

Why int8: the WebGPU `q4f16` export copies sections faithfully, but the CPU `q4` export of this checkpoint paraphrases instead of copying (the stock model's CPU export does not, so it is these weights under the fp32 int4 graph). An int8 CPU file is the candidate replacement for running on the CPU: in Node (`run.mjs --dtype int8`) and the browser's wasm backend. The check cell below says whether it copies.

A CPU runtime is enough; the builder does not use a GPU. Fill in the first cell and run top to bottom.

In [ ]:
HF_USER = "playforgecoding"
MERGED_REPO = f"{HF_USER}/LFM2-1.2B-Extract-lesson"
ONNX_REPO = f"{HF_USER}/LFM2-1.2B-Extract-lesson-ONNX"
RELAYOUT_URL = "https://raw.githubusercontent.com/Spelling-Creator/spelling-creator/main/packages/core/scripts/extract-eval/relayout-onnx.py"

In [ ]:
%pip install -q onnxruntime-genai onnx onnx_ir onnxscript huggingface_hub

## Build the int8 CPU export

In [ ]:
import os, subprocess, sys, urllib.request
from huggingface_hub import snapshot_download, hf_hub_download

snapshot_download(MERGED_REPO, local_dir="merged")
subprocess.run(
    [sys.executable, "-m", "onnxruntime_genai.models.builder",
     "-i", "merged", "-o", "build-cpu-int8", "-p", "int8", "-e", "cpu", "-c", "hf-cache"],
    check=True,
)

# The relayout script from the repo. Upload it beside the notebook with the
# file panel, or let this fetch it from main once the branch carrying it has merged.
if not os.path.exists("relayout-onnx.py"):
    urllib.request.urlretrieve(RELAYOUT_URL, "relayout-onnx.py")

# Start the destination from the ONNX repo's current config.json, so the
# relayout merges the int8 entry into the existing q4 and q4f16 entries rather
# than replacing them.
os.makedirs("onnx-repo", exist_ok=True)
hf_hub_download(ONNX_REPO, "config.json", local_dir="onnx-repo")
subprocess.run([sys.executable, "relayout-onnx.py", "build-cpu-int8", "onnx-repo", "int8", "merged"], check=True)
print(os.listdir("onnx-repo"), os.listdir("onnx-repo/onnx"))

## Check that the int8 export copies rather than paraphrases

Runs the builder's own output through onnxruntime-genai on one section from the held-out lesson. The reply should start with the section's first paragraph copied word for word ("Every pet cat curled up on a sofa today..."), not a summary of it.

In [ ]:
import json
import onnxruntime_genai as og

SCHEMA = json.dumps({
    "type": "object",
    "properties": {
        "name": {"type": "string", "description": "The section heading, or an empty string if there is none."},
        "paragraphs": {"type": "array", "description": "The passage: one item per paragraph, each paragraph copied in full, word for word, starting with the very first paragraph. Paragraphs are separated by blank lines in the document; a paragraph is a whole block of several sentences, never a single sentence.", "items": {"type": "string"}},
        "spellingWords": {"type": "array", "description": "The words on the spelling line, in order, exactly as written.", "items": {"type": "string"}},
        "questions": {"type": "array", "description": "Every question, in order.", "items": {"type": "object", "properties": {
            "prompt": {"type": "string", "description": "The question as written, without its answer."},
            "type": {"type": "string", "enum": ["single", "number", "multiple", "multiple_open", "background", "open", "wyr", "paraphrase"]},
            "answers": {"type": "array", "description": "The answer or answers printed with the question, exactly as written. Empty when none is printed.", "items": {"type": "string"}},
            "steps": {"type": "array", "description": "Numbered working-out lines under a number question. Usually empty.", "items": {"type": "string"}}},
            "required": ["prompt", "type", "answers"]}},
    },
    "required": ["name", "paragraphs", "spellingWords", "questions"],
}, indent=1)

SECTION = """The Wildcat Ancestor

Every pet cat curled up on a sofa today is descended from a single wild species: the AFRICAN WILDCAT, known to scientists as Felis lybica. This sandy-coloured hunter still roams the DESERTS, GRASSLANDS, and PLAINS of North Africa and the Near East. It looks much like a large, long-legged tabby, and it lives alone, hunting by night.

The story of how this shy WILDCAT became a house cat began with farming. When people first stored harvests of grain, the great piles drew MICE, BIRDS, and INSECTS in enormous numbers. The wildcats followed that easy prey into human VILLAGES. Rather than being captured, the boldest cats simply chose to stay, and this slow, self-guided change is called DOMESTICATION.

Spelling words: FELINE, CREATURE, PROWLED, NATURAL

1. From which single wild species is every pet cat descended? (Answer: THE AFRICAN WILDCAT)
2. What is the name for the slow change of a wild animal into a tame one? (Answer: DOMESTICATION)
3. Name a wild animal that hunts.
4. Would you rather hunt by night or by day?
"""

model = og.Model("build-cpu-int8")
tokenizer = og.Tokenizer(model)
prompt = (
    "<|startoftext|><|im_start|>system\nReturn data as a JSON object with the following schema:\n" + SCHEMA +
    "<|im_end|>\n<|im_start|>user\n" + SECTION + "<|im_end|>\n<|im_start|>assistant\n"
)
params = og.GeneratorParams(model)
params.set_search_options(max_length=2500, do_sample=False)
generator = og.Generator(model, params)
generator.append_tokens(tokenizer.encode(prompt))
out = []
while not generator.is_done() and len(out) < 700:
    generator.generate_next_token()
    out.append(generator.get_next_tokens()[0])
reply = tokenizer.decode(out)
print(reply[:1200])
print("\ncopies faithfully:", "Every pet cat curled up on a sofa today is descended" in reply)

If the last line says `False`, stop here and do not upload: the int8 build is no better than q4 for this checkpoint. Otherwise run on.

## Upload

Adds the int8 files and the merged `config.json` to the ONNX repo. The model cards and licence are files in the repo (`model-cards/`) and are pushed with the `hf` CLI, not from here.

## Upload

In [ ]:
from huggingface_hub import login, HfApi

login()
api = HfApi()
api.upload_folder(folder_path="onnx-repo", repo_id=ONNX_REPO, commit_message="int8 CPU export")
print("pushed", ONNX_REPO)